# Inferência — PA1: Segmentação de Instâncias

Aprendizado Profundo · FGV CDIA  
Bruno Ferreira & Elisa Soares

Este notebook carrega o modelo treinado e realiza **inferência em qualquer imagem**, sem necessidade de retreinamento.

Saída: máscara de instâncias colorida + contagem total de núcleos encontrados.

## 1. Configuração — informe o caminho da imagem e do checkpoint

In [ ]:
# ── CONFIGURAÇÃO ──────────────────────────────────────────────
# Altere estas variáveis antes de rodar:

IMAGEM_PATH = "pa1/data/stage1_train/00071198d059ba7f5914a526d124d28e6d010c92466da21d4a04cd5413362552/images/00071198d059ba7f5914a526d124d28e6d010c92466da21d4a04cd5413362552.png"
CHECKPOINT_PATH = "pa1/outputs/checkpoints/parte2_baseline_unet.pt"  # ou pa1/outputs/checkpoints/parte1_baseline_unet.pt

# "trilha_a"  → U-Net 3 classes + Watershed (recomendado)
# "baseline"  → U-Net binária + componentes conexos
MODO = "trilha_a"

## 2. Imports e carregamento do modelo

In [ ]:
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

# Garante que o pacote pa1 é encontrado a partir da raiz do repositório
repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pa1.models import UNet
from pa1.postprocessing import semantic_to_instances, watershed_to_instances

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

# Instancia o modelo correto com base no MODO escolhido
if MODO == "trilha_a":
    out_channels = 3
else:
    out_channels = 2

model = UNet(in_channels=3, out_channels=out_channels).to(device)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt)
model.eval()
print(f"Checkpoint carregado: {CHECKPOINT_PATH} (out_channels={out_channels})")

## 3. Pré-processamento da imagem

In [ ]:
import cv2

# Lê a imagem e converte para RGB
img_bgr = cv2.imread(str(IMAGAGEM_PATH) if False else IMAGAGEM_PATH)
if img_bgr is None:
    img_pil = Image.open(IMAGAGEM_PATH).convert("RGB")
    img_rgb = np.array(img_pil)
else:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print(f"Imagem carregada: {img_rgb.shape} ({img_rgb.dtype})")

# Normalização ImageNet (mesmo esquema do treino com DSB2018)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

img_f = img_rgb.astype(np.float32) / 255.0
img_norm = (img_f - mean) / std

# (H, W, 3) → (1, 3, H, W)
tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
print(f"Tensor de entrada: {tensor.shape}")

## 4. Inferência

In [ ]:
with torch.no_grad():
    logits = model(tensor)

if out_channels >= 3:
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]  # (3, H, W)
    pred_instances = watershed_to_instances(probs)
    print("Decodificação: Watershed (Trilha A)")
else:
    prob_map = torch.softmax(logits, dim=1)[0, 1].cpu().numpy()  # (H, W)
    pred_instances = semantic_to_instances(prob_map)
    print("Decodificação: Componentes Conexos (Baseline Binária)")

n_instancias = len(np.unique(pred_instances[pred_instances > 0]))
print(f"
→ Instâncias detectadas: {n_instancias} núcleos")

## 5. Visualização

In [ ]:
from matplotlib.colors import ListedColormap

def colorir_instancias(inst_map, img_rgb=None, alpha=0.6):
    """Sobrepõe máscara de instâncias coloridas sobre a imagem original."""
    ids = np.unique(inst_map)
    ids = ids[ids > 0]  # remove fundo

    # Cores distintas via colormap
    cmap = plt.cm.get_cmap("tab20", max(len(ids), 1))
    overlay = img_rgb.copy().astype(np.float32) / 255.0 if img_rgb is not None else np.ones((*inst_map.shape, 3), dtype=np.float32)

    patches = []
    for j, inst_id in enumerate(ids):
        mask = inst_map == inst_id
        color = np.array(cmap(j % 20)[:3])
        overlay[mask] = overlay[mask] * (1 - alpha) + color * alpha
        patches.append(mpatches.Patch(color=color, label=f"Núcleo {inst_id}"))

    return np.clip(overlay, 0, 1), patches

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Painel esquerdo: imagem original
axes[0].imshow(img_rgb)
axes[0].set_title("Imagem Original", fontsize=14)
axes[0].axis("off")

# Painel direito: instâncias coloridas
overlaid, patches = colorir_instancias(pred_instances, img_rgb)
axes[1].imshow(overlaid)
axes[1].set_title(f"Instâncias Detectadas: {n_instancias} núcleos", fontsize=14)
axes[1].axis("off")

if n_instancias <= 20:
    axes[1].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)

plt.suptitle(f"PA1 — Segmentação de Instâncias ({MODO})", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("pa1/outputs/inferencia_resultado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva em pa1/outputs/inferencia_resultado.png")